In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_for_lstm_road.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_for_lstm_speed.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_for_lstm_wheel.csv')

In [3]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [4]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

13036
13036
13036


In [5]:
print(X_train[0])
print(X_train[0].size())

tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0')
torch.Size([12288])


In [6]:
print(S_train[0])
print(S_train[0].size())

tensor([59.], device='cuda:0')
torch.Size([1])


In [7]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([0.0053, 0.0053, 0.0053], device='cuda:0')
torch.Size([3])


In [8]:
X_tensor = torch.cat((X_train, S_train), dim=1)

In [9]:
print(X_tensor[0])
print(X_tensor[0].size())

tensor([ 0.,  0.,  0.,  ...,  0.,  0., 59.], device='cuda:0')
torch.Size([12289])


In [16]:
class EnhancedLSTM(nn.Module):
    def __init__(self, input_size=12289, hidden_size=512, num_layers=2, output_size=3, device='cuda'):
        super(EnhancedLSTM, self).__init__()
        self.device = device  # Добавляем параметр устройства
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True).to(device)
        self.fc1 = nn.Linear(hidden_size, 256).to(device)
        self.relu = nn.ReLU().to(device)
        self.fc2 = nn.Linear(256, 128).to(device)
        self.fc3 = nn.Linear(128, output_size).to(device)
        self.tanh = nn.Tanh().to(device)  # Добавляем Tanh как функцию активации для выходного слоя

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        x = hn[-1]
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.tanh(self.fc3(x))  # Применяем Tanh к выходу последнего линейного слоя
        return x

In [29]:
learning_rate = 0.001
batch_size = 64
epochs = 1

# Создание DataLoader
dataset = TensorDataset(X_tensor, y_tensor)  # Предположим, что X_tensor - ваши входные данные, y_tensor - цели
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

# Инициализация модели, функции потерь и оптимизатора
model = EnhancedLSTM().to('cuda')
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Цикл обучения
model.train()
for epoch in range(epochs):
    for inputs, targets in dataloader:
        inputs, targets = inputs.to('cuda'), targets.to('cuda')
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')


C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([64, 3])) that is different to the input size (torch.Size([3])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1/1, Loss: 7.114175969036296e-05


C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([44, 3])) that is different to the input size (torch.Size([3])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [31]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_lstm_3.pth')